In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["PYTHONWARNINGS"] = "ignore"
import sys
import subprocess
from itertools import chain

import numpy as np
import pandas as pd
import scanpy as sc
import loompy as lp
import scipy.sparse as sp
from tqdm import tqdm
from frozendict import frozendict

from pyscenic.utils import modules_from_adjacencies
from pyscenic.aucell import aucell
from ctxcore.genesig import Regulon
from ctxcore.recovery import aucs as calc_aucs, recovery
from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

sc.settings.verbosity = 0
np.random.seed(42)

lambert_df  = pd.read_csv("pyscenic_reference/DatabaseExtract_v_1.01.csv")
curated_tfs = lambert_df.loc[lambert_df["Is TF?"] == "Yes", "HGNC symbol"].tolist()
with open("pyscenic_reference/allTFs_hg38_curated.txt", "w") as f:
    f.write("\n".join(curated_tfs))

In [2]:
INPUT_H5AD    = "checkpoint/S02_NKG2C_gated_for_pyscenic.h5ad"
TFS_PATH      = "pyscenic_reference/allTFs_hg38_curated.txt"
MOTIF_PATH    = "pyscenic_reference/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl"
RANKINGS_GLOB = "pyscenic_reference/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather"
LOOM_INPUT    = "checkpoint/S03_input.loom"
ADJACENCIES   = "checkpoint/S03_adjacencies.csv"
REGULONS      = "checkpoint/S03_regulons.csv"
AUC_MTX       = "results/S03_results09_AUC_matrix.csv"
DE_REGULONS   = "results/S03_results10_DEA_regulon.csv"
OUT_H5AD      = "checkpoint/S03_NKG2C_gated_after_pyscenic.h5ad"

In [3]:
adata = sc.read_h5ad(INPUT_H5AD)
adata = adata[~adata.obs['NK_cluster'].isna()].copy()
adata.obs['NK_cluster'] = adata.obs['NK_cluster'].astype(str).astype('category')

In [4]:
X_csc = adata.X.T.tocsc() if sp.issparse(adata.X) else sp.csc_matrix(adata.X.T)
row_attrs = {"Gene": np.array(adata.var_names)}
col_attrs = {
    "CellID": np.array(adata.obs_names),
    "nGene":  np.array((adata.X > 0).sum(axis=1)).flatten(),
    "nUMI":   np.array(adata.X.sum(axis=1)).flatten()}
lp.create(LOOM_INPUT, X_csc, row_attrs, col_attrs)

In [5]:
cmd = (
    f"pyscenic grn {LOOM_INPUT} {TFS_PATH} "
    f"-o {ADJACENCIES} "
    "--num_workers 8 "
    "--seed 42")
!{cmd}

preparing dask client
parsing input
creating dask graph
8 partitions
computing dask graph
not shutting down client, client was created externally
finished



2026-07-20 23:27:41,807 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2026-07-20 23:27:42,782 - pyscenic.cli.pyscenic - INFO - Inferring regulatory networks.

2026-07-21 00:05:48,813 - pyscenic.cli.pyscenic - INFO - Writing results to file.


In [6]:
def process_module(db, module, annotation_groups,
                   rank_threshold=1500, auc_threshold=0.05, nes_threshold=3.0):
    rank_matrix = db.load(module)
    if rank_matrix.empty:
        return None
    motif_ids, gene_names, rank_values = (
        rank_matrix.index.values,
        rank_matrix.columns.values,
        rank_matrix.values
    )

    gene_weights = np.ones(len(gene_names))
    auc_values   = calc_aucs(rank_matrix, db.total_genes, gene_weights, auc_threshold)
    nes_scores   = (auc_values - auc_values.mean()) / auc_values.std()

    significant_motifs = motif_ids[nes_scores >= nes_threshold]
    if len(significant_motifs) == 0:
        return None

    recovery_curves, _ = recovery(
        rank_matrix, db.total_genes, gene_weights, rank_threshold, auc_threshold, no_auc=True
    )
    recovery_threshold = recovery_curves.mean(axis=0) + 2.0 * recovery_curves.std(axis=0)

    tf_name        = module.transcription_factor
    context        = frozenset(chain(module.context, [db.name]))
    motif_to_index = {motif: index for index, motif in enumerate(motif_ids)}
    result_rows    = []

    for motif_id in significant_motifs:
        if motif_id not in annotation_groups:
            continue
        annotation_rows = annotation_groups[motif_id][
            pd.notnull(annotation_groups[motif_id]['Annotation'])
        ]
        if len(annotation_rows) == 0:
            continue
        best_annotation = annotation_rows.sort_values(
            ['MotifSimilarityQvalue', 'OrthologousIdentity'], ascending=[False, True]
        ).iloc[-1]

        feature_index    = motif_to_index[motif_id]
        recovery_curve   = recovery_curves[feature_index]
        motif_gene_ranks = rank_values[feature_index]
        leading_edge     = recovery_curve > recovery_threshold[:len(recovery_curve)]
        rank_at_max      = (
            int(np.max(np.where(leading_edge)) + 1)
            if np.any(leading_edge) else rank_threshold
        )
        target_genes = [
            (gene, module[gene] if gene in module.gene2weight else 1.0)
            for gene in gene_names[np.where(motif_gene_ranks <= rank_at_max)[0]]
        ]

        result_rows.append({
            ('Enrichment', 'TF'):                    tf_name,
            ('Enrichment', 'MotifID'):                motif_id,
            ('Enrichment', 'AUC'):                    auc_values[feature_index],
            ('Enrichment', 'NES'):                    nes_scores[feature_index],
            ('Enrichment', 'MotifSimilarityQvalue'):  best_annotation.get('MotifSimilarityQvalue', np.nan),
            ('Enrichment', 'OrthologousIdentity'):    best_annotation.get('OrthologousIdentity', np.nan),
            ('Enrichment', 'Annotation'):             best_annotation.get('Annotation', ''),
            ('Enrichment', 'Context'):                context,
            ('Enrichment', 'TargetGenes'):            target_genes,
            ('Enrichment', 'RankAtMax'):               rank_at_max,
        })

    if not result_rows:
        return None
    result = pd.DataFrame(result_rows).set_index(
        [('Enrichment', 'TF'), ('Enrichment', 'MotifID')]
    )
    result.index.names = ['TF', 'MotifID']
    return result


expression_df = pd.DataFrame(
    adata.X.toarray() if sp.issparse(adata.X) else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

adjacency_df      = pd.read_csv(ADJACENCIES)
candidate_modules = list(modules_from_adjacencies(adjacency_df, expression_df, rho_mask_dropouts=True))

ranking_database  = RankingDatabase(fname=RANKINGS_GLOB, name=os.path.basename(RANKINGS_GLOB))
motif_annotation  = pd.read_csv(MOTIF_PATH, sep='\t', low_memory=False).rename(columns={
    '#motif_id':               'MotifID',
    'motif_similarity_qvalue': 'MotifSimilarityQvalue',
    'orthologous_identity':    'OrthologousIdentity',
    'description':             'Annotation',
})
annotation_groups = {motif_id: group for motif_id, group in motif_annotation.groupby('MotifID')}

module_result_list = []
for module in tqdm(candidate_modules, desc="ctx"):
    try:
        module_result = process_module(ranking_database, module, annotation_groups)
        if module_result is not None and len(module_result):
            module_result_list.append(module_result)
    except Exception:
        continue

all_regulons = pd.concat(module_result_list)
all_regulons.to_csv(REGULONS)


2026-07-21 00:05:54,228 - pyscenic.utils - INFO - Calculating Pearson correlations.

2026-07-21 00:05:54,491 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [True].

2026-07-21 00:07:04,346 - pyscenic.utils - INFO - Creating modules.
ctx: 100%|██████████| 4345/4345 [19:10<00:00,  3.78it/s]


In [7]:
regulon_objects = []
for tf_name, tf_group in all_regulons.groupby(level=0):
    gene2weight = {}
    for target_genes in tf_group[('Enrichment', 'TargetGenes')]:
        for gene, weight in target_genes:
            if gene not in gene2weight or weight > gene2weight[gene]:
                gene2weight[gene] = weight
    if gene2weight:
        regulon_objects.append(Regulon(
            name=f"{tf_name}(+)",
            gene2weight=gene2weight,
            gene2occurrence={},
            transcription_factor=tf_name,
        ))

auc_mtx = aucell(expression_df, regulon_objects,
                  auc_threshold=0.05, seed=42, num_workers=1)
auc_mtx.to_csv(AUC_MTX)

100%|██████████| 929/929 [00:58<00:00, 15.78it/s]


In [8]:
nk_clusters = sorted(adata.obs['NK_cluster'].cat.categories.tolist())
test_rows   = []

for cluster in nk_clusters:
    cluster_mask = adata.obs['NK_cluster'].values == cluster
    other_mask   = ~cluster_mask
    for regulon_name in auc_mtx.columns:
        cluster_auc_scores = auc_mtx.loc[adata.obs_names[cluster_mask], regulon_name].values
        other_auc_scores   = auc_mtx.loc[adata.obs_names[other_mask],   regulon_name].values
        if cluster_auc_scores.size < 5 or other_auc_scores.size < 5:
            continue
        test_statistic, p_value = ranksums(cluster_auc_scores, other_auc_scores)
        log2fc = np.log2((cluster_auc_scores.mean() + 1e-9) / (other_auc_scores.mean() + 1e-9))
        test_rows.append({
            "NK_cluster":  cluster,
            "regulon":     regulon_name,
            "mean_in":     cluster_auc_scores.mean(),
            "mean_out":    other_auc_scores.mean(),
            "log2FC":      log2fc,
            "avg_log2FC":  log2fc,
            "statistic":   test_statistic,
            "pval":        p_value,
        })

de_regulons = pd.DataFrame(test_rows)
de_regulons["padj"] = multipletests(de_regulons["pval"], method="fdr_bh")[1]
de_regulons = de_regulons.sort_values(["NK_cluster", "padj"])
de_regulons.to_csv(DE_REGULONS, index=False)

In [9]:
adata.obsm["X_aucell"]     = auc_mtx.loc[adata.obs_names].values
adata.uns["regulon_names"] = list(auc_mtx.columns)
adata.layers["counts"]     = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.write_h5ad(OUT_H5AD)

with open("session/S03_session_info.txt", "w") as f:
    f.write(f"Python {sys.version}\n\n")
    f.write(subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout)